# Subject: test_free_threading
Source: /home/devuser/edu_data/datasets/TrainingDB/EncryT/cpython/Lib/test/test_free_threading

### File: test_bisect.py

In [ ]:
import unittest
from test.support import import_helper, threading_helper
import random

py_bisect = import_helper.import_fresh_module('bisect', blocked=['_bisect'])
c_bisect = import_helper.import_fresh_module('bisect', fresh=['_bisect'])


NTHREADS = 4
OBJECT_COUNT = 500


class TestBase:
    def do_racing_insort(self, insert_method):
        def insert(data):
            for _ in range(OBJECT_COUNT):
                x = random.randint(-OBJECT_COUNT, OBJECT_COUNT)
                insert_method(data, x)

        data = list(range(OBJECT_COUNT))
        threading_helper.run_concurrently(
            worker_func=insert, args=(data,), nthreads=NTHREADS
        )
        if False:
            # These functions are not thread-safe and so the list can become
            # unsorted.  However, we don't want Python to crash if these
            # functions are used concurrently on the same sequence.  This
            # should also not produce any TSAN warnings.
            self.assertTrue(self.is_sorted_ascending(data))

    def test_racing_insert_right(self):
        self.do_racing_insort(self.mod.insort_right)

    def test_racing_insert_left(self):
        self.do_racing_insort(self.mod.insort_left)

    @staticmethod
    def is_sorted_ascending(lst):
        """
        Check if the list is sorted in ascending order (non-decreasing).
        """
        return all(lst[i - 1] <= lst[i] for i in range(1, len(lst)))


@threading_helper.requires_working_threading()
class TestPyBisect(unittest.TestCase, TestBase):
    mod = py_bisect


@threading_helper.requires_working_threading()
class TestCBisect(unittest.TestCase, TestBase):
    mod = c_bisect


if __name__ == "__main__":
    unittest.main()

### File: test_code.py

In [ ]:
import unittest

from threading import Thread
from unittest import TestCase

from test.support import threading_helper

@threading_helper.requires_working_threading()
class TestCode(TestCase):
    def test_code_attrs(self):
        """Test concurrent accesses to lazily initialized code attributes"""
        code_objects = []
        for _ in range(1000):
            code_objects.append(compile("a + b", "<string>", "eval"))

        def run_in_thread():
            for code in code_objects:
                self.assertIsInstance(code.co_code, bytes)
                self.assertIsInstance(code.co_freevars, tuple)
                self.assertIsInstance(code.co_varnames, tuple)

        threads = [Thread(target=run_in_thread) for _ in range(2)]
        for thread in threads:
            thread.start()
        for thread in threads:
            thread.join()


if __name__ == "__main__":
    unittest.main()

### File: test_cprofile.py

In [ ]:
import unittest

from test.support import threading_helper

import cProfile
import pstats


NTHREADS = 10
INSERT_PER_THREAD = 1000


@threading_helper.requires_working_threading()
class TestCProfile(unittest.TestCase):
    def test_cprofile_racing_list_insert(self):
        def list_insert(lst):
            for i in range(INSERT_PER_THREAD):
                lst.insert(0, i)

        lst = []

        with cProfile.Profile() as pr:
            threading_helper.run_concurrently(
                worker_func=list_insert, nthreads=NTHREADS, args=(lst,)
            )
            pr.create_stats()
            ps = pstats.Stats(pr)
            stats_profile = ps.get_stats_profile()
            list_insert_profile = stats_profile.func_profiles[
                "<method 'insert' of 'list' objects>"
            ]
            # Even though there is no explicit recursive call to insert,
            # cProfile may record some calls as recursive due to limitations
            # in its handling of multithreaded programs. This issue is not
            # directly related to FT Python itself; however, it tends to be
            # more noticeable when using FT Python. Therefore, consider only
            # the calls section and disregard the recursive part.
            list_insert_ncalls = list_insert_profile.ncalls.split("/")[0]
            self.assertEqual(
                int(list_insert_ncalls), NTHREADS * INSERT_PER_THREAD
            )

        self.assertEqual(len(lst), NTHREADS * INSERT_PER_THREAD)

### File: test_dbm_gnu.py

In [ ]:
import unittest

from test.support import import_helper, os_helper, threading_helper
from test.support.threading_helper import run_concurrently

import threading

gdbm = import_helper.import_module("dbm.gnu")

NTHREADS = 10
KEY_PER_THREAD = 1000

gdbm_filename = "test_gdbm_file"


@threading_helper.requires_working_threading()
class TestGdbm(unittest.TestCase):
    def test_racing_dbm_gnu(self):
        def gdbm_multi_op_worker(db):
            # Each thread sets, gets, and iterates
            tid = threading.get_ident()

            # Insert keys
            for i in range(KEY_PER_THREAD):
                db[f"key_{tid}_{i}"] = f"value_{tid}_{i}"

            for i in range(KEY_PER_THREAD):
                # Keys and values are stored as bytes; encode values for
                # comparison
                key = f"key_{tid}_{i}"
                value = f"value_{tid}_{i}".encode()
                self.assertIn(key, db)
                self.assertEqual(db[key], value)
                self.assertEqual(db.get(key), value)
                self.assertIsNone(db.get("not_exist"))
                with self.assertRaises(KeyError):
                    db["not_exist"]

            # Iterate over the database keys and verify only those belonging
            # to this thread. Other threads may concurrently delete their keys.
            key_prefix = f"key_{tid}".encode()
            key = db.firstkey()
            key_count = 0
            while key:
                if key.startswith(key_prefix):
                    self.assertIn(key, db)
                    key_count += 1
                key = db.nextkey(key)

            # Can't assert key_count == KEY_PER_THREAD because concurrent
            # threads may insert or delete keys during iteration. This can
            # cause keys to be skipped or counted multiple times, making the
            # count unreliable.
            # See: https://www.gnu.org.ua/software/gdbm/manual/Sequential.html
            # self.assertEqual(key_count, KEY_PER_THREAD)

            # Delete this thread's keys
            for i in range(KEY_PER_THREAD):
                key = f"key_{tid}_{i}"
                del db[key]
                self.assertNotIn(key, db)
                with self.assertRaises(KeyError):
                    del db["not_exist"]

            # Re-insert keys
            for i in range(KEY_PER_THREAD):
                db[f"key_{tid}_{i}"] = f"value_{tid}_{i}"

        with os_helper.temp_dir() as tmpdirname:
            db = gdbm.open(f"{tmpdirname}/{gdbm_filename}", "c")
            run_concurrently(
                worker_func=gdbm_multi_op_worker, nthreads=NTHREADS, args=(db,)
            )
            self.assertEqual(len(db), NTHREADS * KEY_PER_THREAD)
            db.close()


if __name__ == "__main__":
    unittest.main()

### File: test_dict.py

In [ ]:
import gc
import time
import unittest
import weakref

from ast import Or
from functools import partial
from threading import Barrier, Thread
from unittest import TestCase

try:
    import _testcapi
except ImportError:
    _testcapi = None

from test.support import threading_helper


@threading_helper.requires_working_threading()
class TestDict(TestCase):
    def test_racing_creation_shared_keys(self):
        """Verify that creating dictionaries is thread safe when we
        have a type with shared keys"""
        class C(int):
            pass

        self.racing_creation(C)

    def test_racing_creation_no_shared_keys(self):
        """Verify that creating dictionaries is thread safe when we
        have a type with an ordinary dict"""
        self.racing_creation(Or)

    def test_racing_creation_inline_values_invalid(self):
        """Verify that re-creating a dict after we have invalid inline values
        is thread safe"""
        class C:
            pass

        def make_obj():
            a = C()
            # Make object, make inline values invalid, and then delete dict
            a.__dict__ = {}
            del a.__dict__
            return a

        self.racing_creation(make_obj)

    def test_racing_creation_nonmanaged_dict(self):
        """Verify that explicit creation of an unmanaged dict is thread safe
        outside of the normal attribute setting code path"""
        def make_obj():
            def f(): pass
            return f

        def set(func, name, val):
            # Force creation of the dict via PyObject_GenericGetDict
            func.__dict__[name] = val

        self.racing_creation(make_obj, set)

    def racing_creation(self, cls, set=setattr):
        objects = []
        processed = []

        OBJECT_COUNT = 100
        THREAD_COUNT = 10
        CUR = 0

        for i in range(OBJECT_COUNT):
            objects.append(cls())

        def writer_func(name):
            last = -1
            while True:
                if CUR == last:
                    time.sleep(0.001)
                    continue
                elif CUR == OBJECT_COUNT:
                    break

                obj = objects[CUR]
                set(obj, name, name)
                last = CUR
                processed.append(name)

        writers = []
        for x in range(THREAD_COUNT):
            writer = Thread(target=partial(writer_func, f"a{x:02}"))
            writers.append(writer)
            writer.start()

        for i in range(OBJECT_COUNT):
            CUR = i
            while len(processed) != THREAD_COUNT:
                time.sleep(0.001)
            processed.clear()

        CUR = OBJECT_COUNT

        for writer in writers:
            writer.join()

        for obj_idx, obj in enumerate(objects):
            assert (
                len(obj.__dict__) == THREAD_COUNT
            ), f"{len(obj.__dict__)} {obj.__dict__!r} {obj_idx}"
            for i in range(THREAD_COUNT):
                assert f"a{i:02}" in obj.__dict__, f"a{i:02} missing at {obj_idx}"

    def test_racing_set_dict(self):
        """Races assigning to __dict__ should be thread safe"""

        def f(): pass
        l = []
        THREAD_COUNT = 10
        class MyDict(dict): pass

        def writer_func(l):
            for i in range(1000):
                d = MyDict()
                l.append(weakref.ref(d))
                f.__dict__ = d

        lists = []
        writers = []
        for x in range(THREAD_COUNT):
            thread_list = []
            lists.append(thread_list)
            writer = Thread(target=partial(writer_func, thread_list))
            writers.append(writer)

        for writer in writers:
            writer.start()

        for writer in writers:
            writer.join()

        f.__dict__ = {}
        gc.collect()

        for thread_list in lists:
            for ref in thread_list:
                self.assertIsNone(ref())

    def test_racing_get_set_dict(self):
        """Races getting and setting a dict should be thread safe"""
        THREAD_COUNT = 10
        barrier = Barrier(THREAD_COUNT)
        def work(d):
            barrier.wait()
            for _ in range(1000):
                d[10] = 0
                d.get(10, None)
                _ = d[10]

        d = {}
        worker_threads = []
        for ii in range(THREAD_COUNT):
            worker_threads.append(Thread(target=work, args=[d]))
        for t in worker_threads:
            t.start()
        for t in worker_threads:
            t.join()


    def test_racing_set_object_dict(self):
        """Races assigning to __dict__ should be thread safe"""
        class C: pass
        class MyDict(dict): pass
        for cyclic in (False, True):
            f = C()
            f.__dict__ = {"foo": 42}
            THREAD_COUNT = 10

            def writer_func(l):
                for i in range(1000):
                    if cyclic:
                        other_d = {}
                    d = MyDict({"foo": 100})
                    if cyclic:
                        d["x"] = other_d
                        other_d["bar"] = d
                    l.append(weakref.ref(d))
                    f.__dict__ = d

            def reader_func():
                for i in range(1000):
                    f.foo

            lists = []
            readers = []
            writers = []
            for x in range(THREAD_COUNT):
                thread_list = []
                lists.append(thread_list)
                writer = Thread(target=partial(writer_func, thread_list))
                writers.append(writer)

            for x in range(THREAD_COUNT):
                reader = Thread(target=partial(reader_func))
                readers.append(reader)

            for writer in writers:
                writer.start()
            for reader in readers:
                reader.start()

            for writer in writers:
                writer.join()

            for reader in readers:
                reader.join()

            f.__dict__ = {}
            gc.collect()
            gc.collect()

            count = 0
            ids = set()
            for thread_list in lists:
                for i, ref in enumerate(thread_list):
                    if ref() is None:
                        continue
                    count += 1
                    ids.add(id(ref()))
                    count += 1

            self.assertEqual(count, 0)

    def test_racing_object_get_set_dict(self):
        e = Exception()

        def writer():
            for i in range(10000):
                e.__dict__ = {1:2}

        def reader():
            for i in range(10000):
                e.__dict__

        t1 = Thread(target=writer)
        t2 = Thread(target=reader)

        with threading_helper.start_threads([t1, t2]):
            pass

if __name__ == "__main__":
    unittest.main()

### File: test_enumerate.py

In [ ]:
import unittest
import sys
from threading import Thread, Barrier

from test.support import threading_helper

threading_helper.requires_working_threading(module=True)

class EnumerateThreading(unittest.TestCase):

    @threading_helper.reap_threads
    def test_threading(self):
        number_of_threads = 10
        number_of_iterations = 8
        n = 100
        start = sys.maxsize - 40
        barrier = Barrier(number_of_threads)
        def work(enum):
            barrier.wait()
            while True:
                try:
                    _ = next(enum)
                except StopIteration:
                    break

        for it in range(number_of_iterations):
            enum = enumerate(tuple(range(start, start + n)))
            worker_threads = []
            for ii in range(number_of_threads):
                worker_threads.append(
                    Thread(target=work, args=[enum]))
            with threading_helper.start_threads(worker_threads):
                pass

            barrier.reset()

if __name__ == "__main__":
    unittest.main()

### File: test_func_annotations.py

In [ ]:
import concurrent.futures
import unittest
import inspect
from threading import Barrier
from unittest import TestCase

from test.support import threading_helper, Py_GIL_DISABLED

threading_helper.requires_working_threading(module=True)


def get_func_annotation(f, b):
    b.wait()
    return inspect.get_annotations(f)


def get_func_annotation_dunder(f, b):
    b.wait()
    return f.__annotations__


def set_func_annotation(f, b):
    b.wait()
    f.__annotations__ = {'x': int, 'y': int, 'return': int}
    return f.__annotations__


@unittest.skipUnless(Py_GIL_DISABLED, "Enable only in FT build")
class TestFTFuncAnnotations(TestCase):
    NUM_THREADS = 4

    def test_concurrent_read(self):
        def f(x: int) -> int:
            return x + 1

        for _ in range(10):
            with concurrent.futures.ThreadPoolExecutor(max_workers=self.NUM_THREADS) as executor:
                b = Barrier(self.NUM_THREADS)
                futures = {executor.submit(get_func_annotation, f, b): i for i in range(self.NUM_THREADS)}
                for fut in concurrent.futures.as_completed(futures):
                    annotate = fut.result()
                    self.assertIsNotNone(annotate)
                    self.assertEqual(annotate, {'x': int, 'return': int})

            with concurrent.futures.ThreadPoolExecutor(max_workers=self.NUM_THREADS) as executor:
                b = Barrier(self.NUM_THREADS)
                futures = {executor.submit(get_func_annotation_dunder, f, b): i for i in range(self.NUM_THREADS)}
                for fut in concurrent.futures.as_completed(futures):
                    annotate = fut.result()
                    self.assertIsNotNone(annotate)
                    self.assertEqual(annotate, {'x': int, 'return': int})

    def test_concurrent_write(self):
        def bar(x: int, y: float) -> float:
            return y ** x

        for _ in range(10):
            with concurrent.futures.ThreadPoolExecutor(max_workers=self.NUM_THREADS) as executor:
                b = Barrier(self.NUM_THREADS)
                futures = {executor.submit(set_func_annotation, bar, b): i for i in range(self.NUM_THREADS)}
                for fut in concurrent.futures.as_completed(futures):
                    annotate = fut.result()
                    self.assertIsNotNone(annotate)
                    self.assertEqual(annotate, {'x': int, 'y': int, 'return': int})

            # func_get_annotations returns in-place dict, so bar.__annotations__ should be modified as well
            self.assertEqual(bar.__annotations__, {'x': int, 'y': int, 'return': int})

### File: test_functools.py

In [ ]:
import random
import unittest

from functools import lru_cache
from threading import Barrier, Thread

from test.support import threading_helper

@threading_helper.requires_working_threading()
class TestLRUCache(unittest.TestCase):

    def _test_concurrent_operations(self, maxsize):
        num_threads = 10
        b = Barrier(num_threads)
        @lru_cache(maxsize=maxsize)
        def func(arg=0):
            return object()


        def thread_func():
            b.wait()
            for i in range(1000):
                r = random.randint(0, 1000)
                if i < 800:
                    func(i)
                elif i < 900:
                    func.cache_info()
                else:
                    func.cache_clear()

        threads = []
        for i in range(num_threads):
            t = Thread(target=thread_func)
            threads.append(t)

        with threading_helper.start_threads(threads):
            pass

    def test_concurrent_operations_unbounded(self):
        self._test_concurrent_operations(maxsize=None)

    def test_concurrent_operations_bounded(self):
        self._test_concurrent_operations(maxsize=128)

    def _test_reentrant_cache_clear(self, maxsize):
        num_threads = 10
        b = Barrier(num_threads)
        @lru_cache(maxsize=maxsize)
        def func(arg=0):
            func.cache_clear()
            return object()


        def thread_func():
            b.wait()
            for i in range(1000):
                func(random.randint(0, 10000))

        threads = []
        for i in range(num_threads):
            t = Thread(target=thread_func)
            threads.append(t)

        with threading_helper.start_threads(threads):
            pass

    def test_reentrant_cache_clear_unbounded(self):
        self._test_reentrant_cache_clear(maxsize=None)

    def test_reentrant_cache_clear_bounded(self):
        self._test_reentrant_cache_clear(maxsize=128)


if __name__ == "__main__":
    unittest.main()

### File: test_gc.py

In [ ]:
import unittest

import threading
from threading import Thread
from unittest import TestCase
import gc

from test.support import threading_helper


class MyObj:
    pass


@threading_helper.requires_working_threading()
class TestGC(TestCase):
    def test_get_objects(self):
        event = threading.Event()

        def gc_thread():
            for i in range(100):
                o = gc.get_objects()
            event.set()

        def mutator_thread():
            while not event.is_set():
                o1 = MyObj()
                o2 = MyObj()
                o3 = MyObj()
                o4 = MyObj()

        gcs = [Thread(target=gc_thread)]
        mutators = [Thread(target=mutator_thread) for _ in range(4)]
        with threading_helper.start_threads(gcs + mutators):
            pass

    def test_get_referrers(self):
        NUM_GC = 2
        NUM_MUTATORS = 4

        b = threading.Barrier(NUM_GC + NUM_MUTATORS)
        event = threading.Event()

        obj = MyObj()

        def gc_thread():
            b.wait()
            for i in range(100):
                o = gc.get_referrers(obj)
            event.set()

        def mutator_thread():
            b.wait()
            while not event.is_set():
                d1 = { "key": obj }
                d2 = { "key": obj }
                d3 = { "key": obj }
                d4 = { "key": obj }

        gcs = [Thread(target=gc_thread) for _ in range(NUM_GC)]
        mutators = [Thread(target=mutator_thread) for _ in range(NUM_MUTATORS)]
        with threading_helper.start_threads(gcs + mutators):
            pass


if __name__ == "__main__":
    unittest.main()

### File: test_generators.py

In [ ]:
import concurrent.futures
import unittest
from threading import Barrier
from unittest import TestCase
import random
import time

from test.support import threading_helper, Py_GIL_DISABLED

threading_helper.requires_working_threading(module=True)


def random_sleep():
    delay_us = random.randint(50, 100)
    time.sleep(delay_us * 1e-6)

def random_string():
    return ''.join(random.choice('0123456789ABCDEF') for _ in range(10))

def set_gen_name(g, b):
    b.wait()
    random_sleep()
    g.__name__ = random_string()
    return g.__name__

def set_gen_qualname(g, b):
    b.wait()
    random_sleep()
    g.__qualname__ = random_string()
    return g.__qualname__


@unittest.skipUnless(Py_GIL_DISABLED, "Enable only in FT build")
class TestFTGenerators(TestCase):
    NUM_THREADS = 4

    def concurrent_write_with_func(self, func):
        gen = (x for x in range(42))
        for j in range(1000):
            with concurrent.futures.ThreadPoolExecutor(max_workers=self.NUM_THREADS) as executor:
                b = Barrier(self.NUM_THREADS)
                futures = {executor.submit(func, gen, b): i for i in range(self.NUM_THREADS)}
                for fut in concurrent.futures.as_completed(futures):
                    gen_name = fut.result()
                    self.assertEqual(len(gen_name), 10)

    def test_concurrent_write(self):
        with self.subTest(func=set_gen_name):
            self.concurrent_write_with_func(func=set_gen_name)
        with self.subTest(func=set_gen_qualname):
            self.concurrent_write_with_func(func=set_gen_qualname)

### File: test_grp.py

In [ ]:
import unittest

from test.support import import_helper, threading_helper
from test.support.threading_helper import run_concurrently

grp = import_helper.import_module("grp")

from test import test_grp


NTHREADS = 10


@threading_helper.requires_working_threading()
class TestGrp(unittest.TestCase):
    def setUp(self):
        self.test_grp = test_grp.GroupDatabaseTestCase()

    def test_racing_test_values(self):
        # test_grp.test_values() calls grp.getgrall() and checks the entries
        run_concurrently(
            worker_func=self.test_grp.test_values, nthreads=NTHREADS
        )

    def test_racing_test_values_extended(self):
        # test_grp.test_values_extended() calls grp.getgrall(), grp.getgrgid(),
        # grp.getgrnam() and checks the entries
        run_concurrently(
            worker_func=self.test_grp.test_values_extended,
            nthreads=NTHREADS,
        )


if __name__ == "__main__":
    unittest.main()

### File: test_heapq.py

In [ ]:
import unittest

import heapq

from enum import Enum
from threading import Barrier, Lock
from random import shuffle, randint

from test.support import threading_helper
from test.support.threading_helper import run_concurrently
from test import test_heapq


NTHREADS = 10
OBJECT_COUNT = 5_000


class Heap(Enum):
    MIN = 1
    MAX = 2


@threading_helper.requires_working_threading()
class TestHeapq(unittest.TestCase):
    def setUp(self):
        self.test_heapq = test_heapq.TestHeapPython()

    def test_racing_heapify(self):
        heap = list(range(OBJECT_COUNT))
        shuffle(heap)

        run_concurrently(
            worker_func=heapq.heapify, nthreads=NTHREADS, args=(heap,)
        )
        self.test_heapq.check_invariant(heap)

    def test_racing_heappush(self):
        heap = []

        def heappush_func(heap):
            for item in reversed(range(OBJECT_COUNT)):
                heapq.heappush(heap, item)

        run_concurrently(
            worker_func=heappush_func, nthreads=NTHREADS, args=(heap,)
        )
        self.test_heapq.check_invariant(heap)

    def test_racing_heappop(self):
        heap = self.create_heap(OBJECT_COUNT, Heap.MIN)

        # Each thread pops (OBJECT_COUNT / NTHREADS) items
        self.assertEqual(OBJECT_COUNT % NTHREADS, 0)
        per_thread_pop_count = OBJECT_COUNT // NTHREADS

        def heappop_func(heap, pop_count):
            local_list = []
            for _ in range(pop_count):
                item = heapq.heappop(heap)
                local_list.append(item)

            # Each local list should be sorted
            self.assertTrue(self.is_sorted_ascending(local_list))

        run_concurrently(
            worker_func=heappop_func,
            nthreads=NTHREADS,
            args=(heap, per_thread_pop_count),
        )
        self.assertEqual(len(heap), 0)

    def test_racing_heappushpop(self):
        heap = self.create_heap(OBJECT_COUNT, Heap.MIN)
        pushpop_items = self.create_random_list(-5_000, 10_000, OBJECT_COUNT)

        def heappushpop_func(heap, pushpop_items):
            for item in pushpop_items:
                popped_item = heapq.heappushpop(heap, item)
                self.assertTrue(popped_item <= item)

        run_concurrently(
            worker_func=heappushpop_func,
            nthreads=NTHREADS,
            args=(heap, pushpop_items),
        )
        self.assertEqual(len(heap), OBJECT_COUNT)
        self.test_heapq.check_invariant(heap)

    def test_racing_heapreplace(self):
        heap = self.create_heap(OBJECT_COUNT, Heap.MIN)
        replace_items = self.create_random_list(-5_000, 10_000, OBJECT_COUNT)

        def heapreplace_func(heap, replace_items):
            for item in replace_items:
                heapq.heapreplace(heap, item)

        run_concurrently(
            worker_func=heapreplace_func,
            nthreads=NTHREADS,
            args=(heap, replace_items),
        )
        self.assertEqual(len(heap), OBJECT_COUNT)
        self.test_heapq.check_invariant(heap)

    def test_racing_heapify_max(self):
        max_heap = list(range(OBJECT_COUNT))
        shuffle(max_heap)

        run_concurrently(
            worker_func=heapq.heapify_max, nthreads=NTHREADS, args=(max_heap,)
        )
        self.test_heapq.check_max_invariant(max_heap)

    def test_racing_heappush_max(self):
        max_heap = []

        def heappush_max_func(max_heap):
            for item in range(OBJECT_COUNT):
                heapq.heappush_max(max_heap, item)

        run_concurrently(
            worker_func=heappush_max_func, nthreads=NTHREADS, args=(max_heap,)
        )
        self.test_heapq.check_max_invariant(max_heap)

    def test_racing_heappop_max(self):
        max_heap = self.create_heap(OBJECT_COUNT, Heap.MAX)

        # Each thread pops (OBJECT_COUNT / NTHREADS) items
        self.assertEqual(OBJECT_COUNT % NTHREADS, 0)
        per_thread_pop_count = OBJECT_COUNT // NTHREADS

        def heappop_max_func(max_heap, pop_count):
            local_list = []
            for _ in range(pop_count):
                item = heapq.heappop_max(max_heap)
                local_list.append(item)

            # Each local list should be sorted
            self.assertTrue(self.is_sorted_descending(local_list))

        run_concurrently(
            worker_func=heappop_max_func,
            nthreads=NTHREADS,
            args=(max_heap, per_thread_pop_count),
        )
        self.assertEqual(len(max_heap), 0)

    def test_racing_heappushpop_max(self):
        max_heap = self.create_heap(OBJECT_COUNT, Heap.MAX)
        pushpop_items = self.create_random_list(-5_000, 10_000, OBJECT_COUNT)

        def heappushpop_max_func(max_heap, pushpop_items):
            for item in pushpop_items:
                popped_item = heapq.heappushpop_max(max_heap, item)
                self.assertTrue(popped_item >= item)

        run_concurrently(
            worker_func=heappushpop_max_func,
            nthreads=NTHREADS,
            args=(max_heap, pushpop_items),
        )
        self.assertEqual(len(max_heap), OBJECT_COUNT)
        self.test_heapq.check_max_invariant(max_heap)

    def test_racing_heapreplace_max(self):
        max_heap = self.create_heap(OBJECT_COUNT, Heap.MAX)
        replace_items = self.create_random_list(-5_000, 10_000, OBJECT_COUNT)

        def heapreplace_max_func(max_heap, replace_items):
            for item in replace_items:
                heapq.heapreplace_max(max_heap, item)

        run_concurrently(
            worker_func=heapreplace_max_func,
            nthreads=NTHREADS,
            args=(max_heap, replace_items),
        )
        self.assertEqual(len(max_heap), OBJECT_COUNT)
        self.test_heapq.check_max_invariant(max_heap)

    def test_lock_free_list_read(self):
        n, n_threads = 1_000, 10
        l = []
        barrier = Barrier(n_threads * 2)

        count = 0
        lock = Lock()

        def worker():
            with lock:
                nonlocal count
                x = count
                count += 1

            barrier.wait()
            for i in range(n):
                if x % 2:
                    heapq.heappush(l, 1)
                    heapq.heappop(l)
                else:
                    try:
                        l[0]
                    except IndexError:
                        pass

        run_concurrently(worker, n_threads * 2)

    @staticmethod
    def is_sorted_ascending(lst):
        """
        Check if the list is sorted in ascending order (non-decreasing).
        """
        return all(lst[i - 1] <= lst[i] for i in range(1, len(lst)))

    @staticmethod
    def is_sorted_descending(lst):
        """
        Check if the list is sorted in descending order (non-increasing).
        """
        return all(lst[i - 1] >= lst[i] for i in range(1, len(lst)))

    @staticmethod
    def create_heap(size, heap_kind):
        """
        Create a min/max heap where elements are in the range (0, size - 1) and
        shuffled before heapify.
        """
        heap = list(range(OBJECT_COUNT))
        shuffle(heap)
        if heap_kind == Heap.MIN:
            heapq.heapify(heap)
        else:
            heapq.heapify_max(heap)

        return heap

    @staticmethod
    def create_random_list(a, b, size):
        """
        Create a list of random numbers between a and b (inclusive).
        """
        return [randint(-a, b) for _ in range(size)]


if __name__ == "__main__":
    unittest.main()

### File: test_io.py

In [ ]:
import io
import _pyio as pyio
import threading
from unittest import TestCase
from test.support import threading_helper
from random import randint
from sys import getsizeof


class ThreadSafetyMixin:
    # Test pretty much everything that can break under free-threading.
    # Non-deterministic, but at least one of these things will fail if
    # BytesIO object is not free-thread safe.

    def check(self, funcs, *args):
        barrier = threading.Barrier(len(funcs))
        threads = []

        for func in funcs:
            thread = threading.Thread(target=func, args=(barrier, *args))

            threads.append(thread)

        with threading_helper.start_threads(threads):
            pass

    @threading_helper.requires_working_threading()
    @threading_helper.reap_threads
    def test_free_threading(self):
        """Test for segfaults and aborts."""

        def write(barrier, b, *ignore):
            barrier.wait()
            try: b.write(b'0' * randint(100, 1000))
            except ValueError: pass  # ignore write fail to closed file

        def writelines(barrier, b, *ignore):
            barrier.wait()
            b.write(b'0\n' * randint(100, 1000))

        def truncate(barrier, b, *ignore):
            barrier.wait()
            try: b.truncate(0)
            except BufferError: pass  # ignore exported buffer

        def read(barrier, b, *ignore):
            barrier.wait()
            b.read()

        def read1(barrier, b, *ignore):
            barrier.wait()
            b.read1()

        def readline(barrier, b, *ignore):
            barrier.wait()
            b.readline()

        def readlines(barrier, b, *ignore):
            barrier.wait()
            b.readlines()

        def readinto(barrier, b, into, *ignore):
            barrier.wait()
            b.readinto(into)

        def close(barrier, b, *ignore):
            barrier.wait()
            b.close()

        def getvalue(barrier, b, *ignore):
            barrier.wait()
            b.getvalue()

        def getbuffer(barrier, b, *ignore):
            barrier.wait()
            b.getbuffer()

        def iter(barrier, b, *ignore):
            barrier.wait()
            list(b)

        def getstate(barrier, b, *ignore):
            barrier.wait()
            b.__getstate__()

        def setstate(barrier, b, st, *ignore):
            barrier.wait()
            b.__setstate__(st)

        def sizeof(barrier, b, *ignore):
            barrier.wait()
            getsizeof(b)

        self.check([write] * 10, self.ioclass())
        self.check([writelines] * 10, self.ioclass())
        self.check([write] * 10 + [truncate] * 10, self.ioclass())
        self.check([truncate] + [read] * 10, self.ioclass(b'0\n'*204800))
        self.check([truncate] + [read1] * 10, self.ioclass(b'0\n'*204800))
        self.check([truncate] + [readline] * 10, self.ioclass(b'0\n'*20480))
        self.check([truncate] + [readlines] * 10, self.ioclass(b'0\n'*20480))
        self.check([truncate] + [readinto] * 10, self.ioclass(b'0\n'*204800), bytearray(b'0\n'*204800))
        self.check([close] + [write] * 10, self.ioclass())
        self.check([truncate] + [getvalue] * 10, self.ioclass(b'0\n'*204800))
        self.check([truncate] + [getbuffer] * 10, self.ioclass(b'0\n'*204800))
        self.check([truncate] + [iter] * 10, self.ioclass(b'0\n'*20480))
        self.check([truncate] + [getstate] * 10, self.ioclass(b'0\n'*204800))
        state = self.ioclass(b'123').__getstate__()
        self.check([truncate] + [setstate] * 10, self.ioclass(b'0\n'*204800), state)
        self.check([truncate] + [sizeof] * 10, self.ioclass(b'0\n'*204800))

        # no tests for seek or tell because they don't break anything

class CBytesIOTest(ThreadSafetyMixin, TestCase):
    ioclass = io.BytesIO

class PyBytesIOTest(ThreadSafetyMixin, TestCase):
     ioclass = pyio.BytesIO

### File: test_iteration.py

In [ ]:
import threading
import unittest
from test import support

# The race conditions these tests were written for only happen every now and
# then, even with the current numbers. To find rare race conditions, bumping
# these up will help, but it makes the test runtime highly variable under
# free-threading. Overhead is much higher under ThreadSanitizer, but it's
# also much better at detecting certain races, so we don't need as many
# items/threads.
if support.check_sanitizer(thread=True):
    NUMITEMS = 1000
    NUMTHREADS = 2
else:
    NUMITEMS = 100000
    NUMTHREADS = 5
NUMMUTATORS = 2

class ContendedTupleIterationTest(unittest.TestCase):
    def make_testdata(self, n):
        return tuple(range(n))

    def assert_iterator_results(self, results, expected):
        # Most iterators are not atomic (yet?) so they can skip or duplicate
        # items, but they should not invent new items (like the range
        # iterator currently does).
        extra_items = set(results) - set(expected)
        self.assertEqual(set(), extra_items)

    def run_threads(self, func, *args, numthreads=NUMTHREADS):
        threads = []
        for _ in range(numthreads):
            t = threading.Thread(target=func, args=args)
            t.start()
            threads.append(t)
        return threads

    def test_iteration(self):
        """Test iteration over a shared container"""
        seq = self.make_testdata(NUMITEMS)
        results = []
        start = threading.Barrier(NUMTHREADS)
        def worker():
            idx = 0
            start.wait()
            for item in seq:
                idx += 1
            results.append(idx)
        threads = self.run_threads(worker)
        for t in threads:
            t.join()
        # Each thread has its own iterator, so results should be entirely predictable.
        self.assertEqual(results, [NUMITEMS] * NUMTHREADS)

    def test_shared_iterator(self):
        """Test iteration over a shared iterator"""
        seq = self.make_testdata(NUMITEMS)
        it = iter(seq)
        results = []
        start = threading.Barrier(NUMTHREADS)
        def worker():
            items = []
            start.wait()
            # We want a tight loop, so put items in the shared list at the end.
            for item in it:
                items.append(item)
            results.extend(items)
        threads = self.run_threads(worker)
        for t in threads:
            t.join()
        self.assert_iterator_results(results, seq)

class ContendedListIterationTest(ContendedTupleIterationTest):
    def make_testdata(self, n):
        return list(range(n))

    def test_iteration_while_mutating(self):
        """Test iteration over a shared mutating container."""
        seq = self.make_testdata(NUMITEMS)
        results = []
        start = threading.Barrier(NUMTHREADS + NUMMUTATORS)
        endmutate = threading.Event()
        def mutator():
            orig = seq[:]
            # Make changes big enough to cause resizing of the list, with
            # items shifted around for good measure.
            replacement = (orig * 3)[NUMITEMS//2:]
            start.wait()
            while not endmutate.is_set():
                seq.extend(replacement)
                seq[:0] = orig
                seq.__imul__(2)
                seq.extend(seq)
                seq[:] = orig
        def worker():
            items = []
            start.wait()
            # We want a tight loop, so put items in the shared list at the end.
            for item in seq:
                items.append(item)
            results.extend(items)
        mutators = ()
        try:
            threads = self.run_threads(worker)
            mutators = self.run_threads(mutator, numthreads=NUMMUTATORS)
            for t in threads:
                t.join()
        finally:
            endmutate.set()
            for m in mutators:
                m.join()
        self.assert_iterator_results(results, list(seq))


class ContendedRangeIterationTest(ContendedTupleIterationTest):
    def make_testdata(self, n):
        return range(n)

    def assert_iterator_results(self, results, expected):
        # Range iterators that are shared between threads will (right now)
        # sometimes produce items after the end of the range, sometimes
        # _far_ after the end of the range. That should be fixed, but for
        # now, let's just check they're integers that could have resulted
        # from stepping beyond the range bounds.
        extra_items = set(results) - set(expected)
        for item in extra_items:
            self.assertEqual((item - expected.start) % expected.step, 0)

### File: test_itertools.py

In [ ]:
import unittest
from threading import Thread, Barrier
from itertools import batched, chain, cycle
from test.support import threading_helper


threading_helper.requires_working_threading(module=True)

class ItertoolsThreading(unittest.TestCase):

    @threading_helper.reap_threads
    def test_batched(self):
        number_of_threads = 10
        number_of_iterations = 20
        barrier = Barrier(number_of_threads)
        def work(it):
            barrier.wait()
            while True:
                try:
                    next(it)
                except StopIteration:
                    break

        data = tuple(range(1000))
        for it in range(number_of_iterations):
            batch_iterator = batched(data, 2)
            worker_threads = []
            for ii in range(number_of_threads):
                worker_threads.append(
                    Thread(target=work, args=[batch_iterator]))

            with threading_helper.start_threads(worker_threads):
                pass

            barrier.reset()

    @threading_helper.reap_threads
    def test_cycle(self):
        number_of_threads = 6
        number_of_iterations = 10
        number_of_cycles = 400

        barrier = Barrier(number_of_threads)
        def work(it):
            barrier.wait()
            for _ in range(number_of_cycles):
                try:
                    next(it)
                except StopIteration:
                    pass

        data = (1, 2, 3, 4)
        for it in range(number_of_iterations):
            cycle_iterator = cycle(data)
            worker_threads = []
            for ii in range(number_of_threads):
                worker_threads.append(
                    Thread(target=work, args=[cycle_iterator]))

            with threading_helper.start_threads(worker_threads):
                pass

            barrier.reset()

    @threading_helper.reap_threads
    def test_chain(self):
        number_of_threads = 6
        number_of_iterations = 20

        barrier = Barrier(number_of_threads)
        def work(it):
            barrier.wait()
            while True:
                try:
                    next(it)
                except StopIteration:
                    break

        data = [(1, )] * 200
        for it in range(number_of_iterations):
            chain_iterator = chain(*data)
            worker_threads = []
            for ii in range(number_of_threads):
                worker_threads.append(
                    Thread(target=work, args=[chain_iterator]))

            with threading_helper.start_threads(worker_threads):
                pass

            barrier.reset()



if __name__ == "__main__":
    unittest.main()

### File: test_itertools_combinatoric.py

In [ ]:
import unittest
from threading import Thread, Barrier
from itertools import combinations, product
from test.support import threading_helper


threading_helper.requires_working_threading(module=True)

def test_concurrent_iteration(iterator, number_of_threads):
    barrier = Barrier(number_of_threads)
    def iterator_worker(it):
        barrier.wait()
        while True:
            try:
                _ = next(it)
            except StopIteration:
                return

    worker_threads = []
    for ii in range(number_of_threads):
        worker_threads.append(
            Thread(target=iterator_worker, args=[iterator]))

    with threading_helper.start_threads(worker_threads):
        pass

    barrier.reset()

class ItertoolsThreading(unittest.TestCase):

    @threading_helper.reap_threads
    def test_combinations(self):
        number_of_threads = 10
        number_of_iterations = 24

        for it in range(number_of_iterations):
            iterator = combinations((1, 2, 3, 4, 5), 2)
            test_concurrent_iteration(iterator, number_of_threads)

    @threading_helper.reap_threads
    def test_product(self):
        number_of_threads = 10
        number_of_iterations = 24

        for it in range(number_of_iterations):
            iterator = product((1, 2, 3, 4, 5), (10, 20, 30))
            test_concurrent_iteration(iterator, number_of_threads)


if __name__ == "__main__":
    unittest.main()

### File: test_json.py

In [ ]:
from threading import Barrier, Thread
from test.test_json import CTest
from test.support import threading_helper


def encode_json_helper(
    json, worker, data, number_of_threads=12, number_of_json_encodings=100
):
    worker_threads = []
    barrier = Barrier(number_of_threads)
    for index in range(number_of_threads):
        worker_threads.append(
            Thread(target=worker, args=[barrier, data, index])
        )
    for t in worker_threads:
        t.start()
    for ii in range(number_of_json_encodings):
        json.dumps(data)
    data.clear()
    for t in worker_threads:
        t.join()


class MyMapping(dict):
    def __init__(self):
        self.mapping = []

    def items(self):
        return self.mapping


@threading_helper.reap_threads
@threading_helper.requires_working_threading()
class TestJsonEncoding(CTest):
    # Test encoding json with concurrent threads modifying the data cannot
    # corrupt the interpreter

    def test_json_mutating_list(self):
        def worker(barrier, data, index):
            barrier.wait()
            while data:
                for d in data:
                    if len(d) > 5:
                        d.clear()
                    else:
                        d.append(index)

        data = [[], []]
        encode_json_helper(self.json, worker, data)

    def test_json_mutating_exact_dict(self):
        def worker(barrier, data, index):
            barrier.wait()
            while data:
                for d in data:
                    if len(d) > 5:
                        try:
                            key = list(d)[0]
                            d.pop(key)
                        except (KeyError, IndexError):
                            pass
                    else:
                        d[index] = index

        data = [{}, {}]
        encode_json_helper(self.json, worker, data)

    def test_json_mutating_mapping(self):
        def worker(barrier, data, index):
            barrier.wait()
            while data:
                for d in data:
                    if len(d.mapping) > 3:
                        d.mapping.clear()
                    else:
                        d.mapping.append((index, index))

        data = [MyMapping(), MyMapping()]
        encode_json_helper(self.json, worker, data)

### File: test_list.py

In [ ]:
import unittest

from threading import Thread, Barrier
from unittest import TestCase

from test.support import threading_helper


NTHREAD = 10
OBJECT_COUNT = 5_000


class C:
    def __init__(self, v):
        self.v = v


@threading_helper.requires_working_threading()
class TestList(TestCase):
    def test_racing_iter_append(self):
        l = []

        barrier = Barrier(NTHREAD + 1)
        def writer_func(l):
            barrier.wait()
            for i in range(OBJECT_COUNT):
                l.append(C(i + OBJECT_COUNT))

        def reader_func(l):
            barrier.wait()
            while True:
                count = len(l)
                for i, x in enumerate(l):
                    self.assertEqual(x.v, i + OBJECT_COUNT)
                if count == OBJECT_COUNT:
                    break

        writer = Thread(target=writer_func, args=(l,))
        readers = []
        for x in range(NTHREAD):
            reader = Thread(target=reader_func, args=(l,))
            readers.append(reader)
            reader.start()

        writer.start()
        writer.join()
        for reader in readers:
            reader.join()

    def test_racing_iter_extend(self):
        l = []

        barrier = Barrier(NTHREAD + 1)
        def writer_func():
            barrier.wait()
            for i in range(OBJECT_COUNT):
                l.extend([C(i + OBJECT_COUNT)])

        def reader_func():
            barrier.wait()
            while True:
                count = len(l)
                for i, x in enumerate(l):
                    self.assertEqual(x.v, i + OBJECT_COUNT)
                if count == OBJECT_COUNT:
                    break

        writer = Thread(target=writer_func)
        readers = []
        for x in range(NTHREAD):
            reader = Thread(target=reader_func)
            readers.append(reader)
            reader.start()

        writer.start()
        writer.join()
        for reader in readers:
            reader.join()

    def test_store_list_int(self):
        def copy_back_and_forth(b, l):
            b.wait()
            for _ in range(100):
                l[0] = l[1]
                l[1] = l[0]

        l = [0, 1]
        barrier = Barrier(NTHREAD)
        threads = [Thread(target=copy_back_and_forth, args=(barrier, l))
                   for _ in range(NTHREAD)]
        with threading_helper.start_threads(threads):
            pass


if __name__ == "__main__":
    unittest.main()

### File: test_methodcaller.py

In [ ]:
import unittest
from threading import Thread
from operator import methodcaller


class TestMethodcaller(unittest.TestCase):
    def test_methodcaller_threading(self):
        number_of_threads = 10
        size = 4_000

        mc = methodcaller("append", 2)

        def work(mc, l, ii):
            for _ in range(ii):
                mc(l)

        worker_threads = []
        lists = []
        for ii in range(number_of_threads):
            l = []
            lists.append(l)
            worker_threads.append(Thread(target=work, args=[mc, l, size]))
        for t in worker_threads:
            t.start()
        for t in worker_threads:
            t.join()
        for l in lists:
            assert len(l) == size


if __name__ == "__main__":
    unittest.main()

### File: test_mmap.py

In [ ]:
import unittest

from test.support import import_helper, threading_helper
from test.support.threading_helper import run_concurrently

import os
import string
import tempfile
import threading

from collections import Counter

mmap = import_helper.import_module("mmap")

NTHREADS = 10
ANONYMOUS_MEM = -1


@threading_helper.requires_working_threading()
class MmapTests(unittest.TestCase):
    def test_read_and_read_byte(self):
        ascii_uppercase = string.ascii_uppercase.encode()
        # Choose a total mmap size that evenly divides across threads and the
        # read pattern (3 bytes per loop).
        mmap_size = 3 * NTHREADS * len(ascii_uppercase)
        num_bytes_to_read_per_thread = mmap_size // NTHREADS
        bytes_read_from_mmap = []

        def read(mm_obj):
            nread = 0
            while nread < num_bytes_to_read_per_thread:
                b = mm_obj.read_byte()
                bytes_read_from_mmap.append(b)
                b = mm_obj.read(2)
                bytes_read_from_mmap.extend(b)
                nread += 3

        with mmap.mmap(ANONYMOUS_MEM, mmap_size) as mm_obj:
            for i in range(mmap_size // len(ascii_uppercase)):
                mm_obj.write(ascii_uppercase)

            mm_obj.seek(0)
            run_concurrently(
                worker_func=read,
                args=(mm_obj,),
                nthreads=NTHREADS,
            )

        self.assertEqual(len(bytes_read_from_mmap), mmap_size)
        # Count each letter/byte to verify read correctness
        counter = Counter(bytes_read_from_mmap)
        self.assertEqual(len(counter), len(ascii_uppercase))
        # Each letter/byte should be read (3 * NTHREADS) times
        for letter in ascii_uppercase:
            self.assertEqual(counter[letter], 3 * NTHREADS)

    def test_readline(self):
        num_lines = 1000
        lines_read_from_mmap = []
        expected_lines = []

        def readline(mm_obj):
            for i in range(num_lines // NTHREADS):
                line = mm_obj.readline()
                lines_read_from_mmap.append(line)

        # Allocate mmap enough for num_lines (max line 5 bytes including NL)
        with mmap.mmap(ANONYMOUS_MEM, num_lines * 5) as mm_obj:
            for i in range(num_lines):
                line = b"%d\n" % i
                mm_obj.write(line)
                expected_lines.append(line)

            mm_obj.seek(0)
            run_concurrently(
                worker_func=readline,
                args=(mm_obj,),
                nthreads=NTHREADS,
            )

        self.assertEqual(len(lines_read_from_mmap), num_lines)
        # Every line should be read once by threads; order is non-deterministic
        # Sort numerically by integer value
        lines_read_from_mmap.sort(key=lambda x: int(x))
        self.assertEqual(lines_read_from_mmap, expected_lines)

    def test_write_and_write_byte(self):
        thread_letters = list(string.ascii_uppercase)
        self.assertLessEqual(NTHREADS, len(thread_letters))
        per_thread_write_loop = 100

        def write(mm_obj):
            # Each thread picks a unique letter to write
            thread_letter = thread_letters.pop(0)
            thread_bytes = (thread_letter * 2).encode()
            for _ in range(per_thread_write_loop):
                mm_obj.write_byte(thread_bytes[0])
                mm_obj.write(thread_bytes)

        with mmap.mmap(
            ANONYMOUS_MEM, per_thread_write_loop * 3 * NTHREADS
        ) as mm_obj:
            run_concurrently(
                worker_func=write,
                args=(mm_obj,),
                nthreads=NTHREADS,
            )
            mm_obj.seek(0)
            data = mm_obj.read()
            self.assertEqual(len(data), NTHREADS * per_thread_write_loop * 3)
            counter = Counter(data)
            self.assertEqual(len(counter), NTHREADS)
            # Each thread letter should be written `per_thread_write_loop` * 3
            for letter in counter:
                self.assertEqual(counter[letter], per_thread_write_loop * 3)

    def test_move(self):
        ascii_uppercase = string.ascii_uppercase.encode()
        num_letters = len(ascii_uppercase)

        def move(mm_obj):
            for i in range(num_letters):
                # Move 1 byte from the first half to the second half
                mm_obj.move(0 + i, num_letters + i, 1)

        with mmap.mmap(ANONYMOUS_MEM, 2 * num_letters) as mm_obj:
            mm_obj.write(ascii_uppercase)
            run_concurrently(
                worker_func=move,
                args=(mm_obj,),
                nthreads=NTHREADS,
            )

    def test_seek_and_tell(self):
        seek_per_thread = 10

        def seek(mm_obj):
            self.assertTrue(mm_obj.seekable())
            for _ in range(seek_per_thread):
                before_seek = mm_obj.tell()
                mm_obj.seek(1, os.SEEK_CUR)
                self.assertLess(before_seek, mm_obj.tell())

        with mmap.mmap(ANONYMOUS_MEM, 1024) as mm_obj:
            run_concurrently(
                worker_func=seek,
                args=(mm_obj,),
                nthreads=NTHREADS,
            )
            # Each thread seeks from current position, the end position should
            # be the sum of all seeks from all threads.
            self.assertEqual(mm_obj.tell(), NTHREADS * seek_per_thread)

    def test_slice_update_and_slice_read(self):
        thread_letters = list(string.ascii_uppercase)
        self.assertLessEqual(NTHREADS, len(thread_letters))

        def slice_update_and_slice_read(mm_obj):
            # Each thread picks a unique letter to write
            thread_letter = thread_letters.pop(0)
            thread_bytes = (thread_letter * 1024).encode()
            for _ in range(100):
                mm_obj[:] = thread_bytes
                read_bytes = mm_obj[:]
                # Read bytes should be all the same letter, showing no
                # interleaving
                self.assertTrue(all_same(read_bytes))

        with mmap.mmap(ANONYMOUS_MEM, 1024) as mm_obj:
            run_concurrently(
                worker_func=slice_update_and_slice_read,
                args=(mm_obj,),
                nthreads=NTHREADS,
            )

    def test_item_update_and_item_read(self):
        thread_indexes = [i for i in range(NTHREADS)]

        def item_update_and_item_read(mm_obj):
            # Each thread picks a unique index to write
            thread_index = thread_indexes.pop()
            for i in range(100):
                mm_obj[thread_index] = i
                self.assertEqual(mm_obj[thread_index], i)

            # Read values set by other threads, all values
            # should be less than '100'
            for val in mm_obj:
                self.assertLess(int.from_bytes(val), 100)

        with mmap.mmap(ANONYMOUS_MEM, NTHREADS + 1) as mm_obj:
            run_concurrently(
                worker_func=item_update_and_item_read,
                args=(mm_obj,),
                nthreads=NTHREADS,
            )

    @unittest.skipUnless(os.name == "posix", "requires Posix")
    @unittest.skipUnless(hasattr(mmap.mmap, "resize"), "requires mmap.resize")
    def test_resize_and_size(self):
        thread_indexes = [i for i in range(NTHREADS)]

        def resize_and_item_update(mm_obj):
            # Each thread picks a unique index to write
            thread_index = thread_indexes.pop()
            mm_obj.resize(2048)
            self.assertEqual(mm_obj.size(), 2048)
            for i in range(100):
                mm_obj[thread_index] = i
                self.assertEqual(mm_obj[thread_index], i)

        with mmap.mmap(ANONYMOUS_MEM, 1024, flags=mmap.MAP_PRIVATE) as mm_obj:
            run_concurrently(
                worker_func=resize_and_item_update,
                args=(mm_obj,),
                nthreads=NTHREADS,
            )

    def test_close_and_closed(self):
        def close_mmap(mm_obj):
            mm_obj.close()
            self.assertTrue(mm_obj.closed)

        with mmap.mmap(ANONYMOUS_MEM, 1) as mm_obj:
            run_concurrently(
                worker_func=close_mmap,
                args=(mm_obj,),
                nthreads=NTHREADS,
            )

    def test_find_and_rfind(self):
        per_thread_loop = 10

        def find_and_rfind(mm_obj):
            pattern = b'Thread-Ident:"%d"' % threading.get_ident()
            mm_obj.write(pattern)
            for _ in range(per_thread_loop):
                found_at = mm_obj.find(pattern, 0)
                self.assertNotEqual(found_at, -1)
                # Should not find it after the `found_at`
                self.assertEqual(mm_obj.find(pattern, found_at + 1), -1)
                found_at_rev = mm_obj.rfind(pattern, 0)
                self.assertEqual(found_at, found_at_rev)
                # Should not find it after the `found_at`
                self.assertEqual(mm_obj.rfind(pattern, found_at + 1), -1)

        with mmap.mmap(ANONYMOUS_MEM, 1024) as mm_obj:
            run_concurrently(
                worker_func=find_and_rfind,
                args=(mm_obj,),
                nthreads=NTHREADS,
            )

    @unittest.skipUnless(os.name == "posix", "requires Posix")
    @unittest.skipUnless(hasattr(mmap.mmap, "resize"), "requires mmap.resize")
    def test_flush(self):
        mmap_filename = "test_mmap_file"
        resize_to = 1024

        def resize_and_flush(mm_obj):
            mm_obj.resize(resize_to)
            mm_obj.flush()

        with tempfile.TemporaryDirectory() as tmpdirname:
            file_path = f"{tmpdirname}/{mmap_filename}"
            with open(file_path, "wb+") as file:
                file.write(b"CPython")
                file.flush()
                with mmap.mmap(file.fileno(), 1) as mm_obj:
                    run_concurrently(
                        worker_func=resize_and_flush,
                        args=(mm_obj,),
                        nthreads=NTHREADS,
                    )

            self.assertEqual(os.path.getsize(file_path), resize_to)

    def test_mmap_export_as_memoryview(self):
        """
        Each thread creates a memoryview and updates the internal state of the
        mmap object.
        """
        buffer_size = 42

        def create_memoryview_from_mmap(mm_obj):
            memoryviews = []
            for _ in range(100):
                mv = memoryview(mm_obj)
                memoryviews.append(mv)
                self.assertEqual(len(mv), buffer_size)
                self.assertEqual(mv[:7], b"CPython")

            # Cannot close the mmap while it is exported as buffers
            with self.assertRaisesRegex(
                BufferError, "cannot close exported pointers exist"
            ):
                mm_obj.close()

        with mmap.mmap(ANONYMOUS_MEM, 42) as mm_obj:
            mm_obj.write(b"CPython")
            run_concurrently(
                worker_func=create_memoryview_from_mmap,
                args=(mm_obj,),
                nthreads=NTHREADS,
            )
            # Implicit mm_obj.close() verifies all exports (memoryviews) are
            # properly freed.


def all_same(lst):
    return all(item == lst[0] for item in lst)


if __name__ == "__main__":
    unittest.main()

### File: test_monitoring.py

In [ ]:
"""Tests monitoring, sys.settrace, and sys.setprofile in a multi-threaded
environment to verify things are thread-safe in a free-threaded build"""

import sys
import threading
import time
import unittest
import weakref

from contextlib import contextmanager
from sys import monitoring
from test.support import threading_helper
from threading import Thread, _PyRLock, Barrier
from unittest import TestCase


class InstrumentationMultiThreadedMixin:
    thread_count = 10
    func_count = 50
    fib = 12

    def after_threads(self):
        """Runs once after all the threads have started"""
        pass

    def during_threads(self):
        """Runs repeatedly while the threads are still running"""
        pass

    def work(self, n, funcs):
        """Fibonacci function which also calls a bunch of random functions"""
        for func in funcs:
            func()
        if n < 2:
            return n
        return self.work(n - 1, funcs) + self.work(n - 2, funcs)

    def start_work(self, n, funcs):
        # With the GIL builds we need to make sure that the hooks have
        # a chance to run as it's possible to run w/o releasing the GIL.
        time.sleep(0.1)
        self.work(n, funcs)

    def after_test(self):
        """Runs once after the test is done"""
        pass

    def test_instrumentation(self):
        # Setup a bunch of functions which will need instrumentation...
        funcs = []
        for i in range(self.func_count):
            x = {}
            exec("def f(): pass", x)
            funcs.append(x["f"])

        threads = []
        for i in range(self.thread_count):
            # Each thread gets a copy of the func list to avoid contention
            t = Thread(target=self.start_work, args=(self.fib, list(funcs)))
            t.start()
            threads.append(t)

        self.after_threads()

        while True:
            any_alive = False
            for t in threads:
                if t.is_alive():
                    any_alive = True
                    break

            if not any_alive:
                break

            self.during_threads()

        self.after_test()


class MonitoringTestMixin:
    def setUp(self):
        for i in range(6):
            if monitoring.get_tool(i) is None:
                self.tool_id = i
                monitoring.use_tool_id(i, self.__class__.__name__)
                break

    def tearDown(self):
        monitoring.free_tool_id(self.tool_id)


@threading_helper.requires_working_threading()
class SetPreTraceMultiThreaded(InstrumentationMultiThreadedMixin, TestCase):
    """Sets tracing one time after the threads have started"""

    def setUp(self):
        super().setUp()
        self.called = False

    def after_test(self):
        self.assertTrue(self.called)

    def trace_func(self, frame, event, arg):
        self.called = True
        return self.trace_func

    def after_threads(self):
        sys.settrace(self.trace_func)


@threading_helper.requires_working_threading()
class MonitoringMultiThreaded(
    MonitoringTestMixin, InstrumentationMultiThreadedMixin, TestCase
):
    """Uses sys.monitoring and repeatedly toggles instrumentation on and off"""

    def setUp(self):
        super().setUp()
        self.set = False
        self.called = False
        monitoring.register_callback(
            self.tool_id, monitoring.events.LINE, self.callback
        )

    def tearDown(self):
        monitoring.set_events(self.tool_id, 0)
        super().tearDown()

    def callback(self, *args):
        self.called = True

    def after_test(self):
        self.assertTrue(self.called)

    def during_threads(self):
        if self.set:
            monitoring.set_events(
                self.tool_id, monitoring.events.CALL | monitoring.events.LINE
            )
        else:
            monitoring.set_events(self.tool_id, 0)
        self.set = not self.set


@threading_helper.requires_working_threading()
class SetTraceMultiThreaded(InstrumentationMultiThreadedMixin, TestCase):
    """Uses sys.settrace and repeatedly toggles instrumentation on and off"""

    def setUp(self):
        self.set = False
        self.called = False

    def after_test(self):
        self.assertTrue(self.called)

    def tearDown(self):
        sys.settrace(None)

    def trace_func(self, frame, event, arg):
        self.called = True
        return self.trace_func

    def during_threads(self):
        if self.set:
            sys.settrace(self.trace_func)
        else:
            sys.settrace(None)
        self.set = not self.set


@threading_helper.requires_working_threading()
class SetProfileMultiThreaded(InstrumentationMultiThreadedMixin, TestCase):
    """Uses sys.setprofile and repeatedly toggles instrumentation on and off"""

    def setUp(self):
        self.set = False
        self.called = False

    def after_test(self):
        self.assertTrue(self.called)

    def tearDown(self):
        sys.setprofile(None)

    def trace_func(self, frame, event, arg):
        self.called = True
        return self.trace_func

    def during_threads(self):
        if self.set:
            sys.setprofile(self.trace_func)
        else:
            sys.setprofile(None)
        self.set = not self.set


@threading_helper.requires_working_threading()
class SetProfileAllThreadsMultiThreaded(InstrumentationMultiThreadedMixin, TestCase):
    """Uses threading.setprofile_all_threads and repeatedly toggles instrumentation on and off"""

    def setUp(self):
        self.set = False
        self.called = False

    def after_test(self):
        self.assertTrue(self.called)

    def tearDown(self):
        threading.setprofile_all_threads(None)

    def trace_func(self, frame, event, arg):
        self.called = True
        return self.trace_func

    def during_threads(self):
        if self.set:
            threading.setprofile_all_threads(self.trace_func)
        else:
            threading.setprofile_all_threads(None)
        self.set = not self.set


class SetProfileAllMultiThreaded(TestCase):
    def test_profile_all_threads(self):
        done = threading.Event()

        def func():
            pass

        def bg_thread():
            while not done.is_set():
                func()
                func()
                func()
                func()
                func()

        def my_profile(frame, event, arg):
            return None

        bg_threads = []
        for i in range(10):
            t = threading.Thread(target=bg_thread)
            t.start()
            bg_threads.append(t)

        for i in range(100):
            threading.setprofile_all_threads(my_profile)
            threading.setprofile_all_threads(None)

        done.set()
        for t in bg_threads:
            t.join()


class TraceBuf:
    def __init__(self):
        self.traces = []
        self.traces_lock = threading.Lock()

    def append(self, trace):
        with self.traces_lock:
            self.traces.append(trace)


@threading_helper.requires_working_threading()
class MonitoringMisc(MonitoringTestMixin, TestCase):
    def register_callback(self, barrier):
        barrier.wait()

        def callback(*args):
            pass

        for i in range(200):
            monitoring.register_callback(self.tool_id, monitoring.events.LINE, callback)

        self.refs.append(weakref.ref(callback))

    def test_register_callback(self):
        self.refs = []
        threads = []
        barrier = Barrier(5)
        for i in range(5):
            t = Thread(target=self.register_callback, args=(barrier,))
            t.start()
            threads.append(t)

        for thread in threads:
            thread.join()

        monitoring.register_callback(self.tool_id, monitoring.events.LINE, None)
        for ref in self.refs:
            self.assertEqual(ref(), None)

    def test_set_local_trace_opcodes(self):
        def trace(frame, event, arg):
            frame.f_trace_opcodes = True
            return trace

        loops = 1_000

        sys.settrace(trace)
        try:
            l = _PyRLock()

            def f():
                for i in range(loops):
                    with l:
                        pass

            t = Thread(target=f)
            t.start()
            for i in range(loops):
                with l:
                    pass
            t.join()
        finally:
            sys.settrace(None)

    def test_toggle_setprofile_no_new_events(self):
        # gh-136396: Make sure that profile functions are called for newly
        # created threads when profiling is toggled but the set of monitoring
        # events doesn't change
        traces = []

        def profiler(frame, event, arg):
            traces.append((frame.f_code.co_name, event, arg))

        def a(x, y):
            return b(x, y)

        def b(x, y):
            return max(x, y)

        sys.setprofile(profiler)
        try:
            a(1, 2)
        finally:
            sys.setprofile(None)
        traces.clear()

        def thread_main(x, y):
            sys.setprofile(profiler)
            try:
                a(x, y)
            finally:
                sys.setprofile(None)
        t = Thread(target=thread_main, args=(100, 200))
        t.start()
        t.join()

        expected = [
            ("a", "call", None),
            ("b", "call", None),
            ("b", "c_call", max),
            ("b", "c_return", max),
            ("b", "return", 200),
            ("a", "return", 200),
            ("thread_main", "c_call", sys.setprofile),
        ]
        self.assertEqual(traces, expected)

    def observe_threads(self, observer, buf):
        def in_child(ident):
            return ident

        def child(ident):
            with observer():
                in_child(ident)

        def in_parent(ident):
            return ident

        def parent(barrier, ident):
            barrier.wait()
            with observer():
                t = Thread(target=child, args=(ident,))
                t.start()
                t.join()
                in_parent(ident)

        num_threads = 5
        barrier = Barrier(num_threads)
        threads = []
        for i in range(num_threads):
            t = Thread(target=parent, args=(barrier, i))
            t.start()
            threads.append(t)
        for t in threads:
            t.join()

        for i in range(num_threads):
            self.assertIn(("in_parent", "return", i), buf.traces)
            self.assertIn(("in_child", "return", i), buf.traces)

    def test_profile_threads(self):
        buf = TraceBuf()

        def profiler(frame, event, arg):
            buf.append((frame.f_code.co_name, event, arg))

        @contextmanager
        def profile():
            sys.setprofile(profiler)
            try:
                yield
            finally:
                sys.setprofile(None)

        self.observe_threads(profile, buf)

    def test_trace_threads(self):
        buf = TraceBuf()

        def tracer(frame, event, arg):
            buf.append((frame.f_code.co_name, event, arg))
            return tracer

        @contextmanager
        def trace():
            sys.settrace(tracer)
            try:
                yield
            finally:
                sys.settrace(None)

        self.observe_threads(trace, buf)

    def test_monitor_threads(self):
        buf = TraceBuf()

        def monitor_py_return(code, off, retval):
            buf.append((code.co_name, "return", retval))

        monitoring.register_callback(
            self.tool_id, monitoring.events.PY_RETURN, monitor_py_return
        )

        monitoring.set_events(
            self.tool_id, monitoring.events.PY_RETURN
        )

        @contextmanager
        def noop():
            yield

        self.observe_threads(noop, buf)

    def test_trace_concurrent(self):
        # Test calling a function concurrently from a tracing and a non-tracing
        # thread
        b = threading.Barrier(2)

        def func():
            for _ in range(100):
                pass

        def noop():
            pass

        def bg_thread():
            b.wait()
            func()  # this may instrument `func`

        def tracefunc(frame, event, arg):
            # These calls run under tracing can race with the background thread
            for _ in range(10):
                func()
            return tracefunc

        t = Thread(target=bg_thread)
        t.start()
        try:
            sys.settrace(tracefunc)
            b.wait()
            noop()
        finally:
            sys.settrace(None)
        t.join()


if __name__ == "__main__":
    unittest.main()

### File: test_pwd.py

In [ ]:
import unittest

from test.support import threading_helper
from test.support.threading_helper import run_concurrently

from test import test_pwd


NTHREADS = 10


@threading_helper.requires_working_threading()
class TestPwd(unittest.TestCase):
    def setUp(self):
        self.test_pwd = test_pwd.PwdTest()

    def test_racing_test_values(self):
        # test_pwd.test_values() calls pwd.getpwall() and checks the entries
        run_concurrently(
            worker_func=self.test_pwd.test_values, nthreads=NTHREADS
        )

    def test_racing_test_values_extended(self):
        # test_pwd.test_values_extended() calls pwd.getpwall(), pwd.getpwnam(),
        # pwd.getpwduid() and checks the entries
        run_concurrently(
            worker_func=self.test_pwd.test_values_extended,
            nthreads=NTHREADS,
        )


if __name__ == "__main__":
    unittest.main()

### File: test_races.py

In [ ]:
# It's most useful to run these tests with ThreadSanitizer enabled.
import sys
import functools
import threading
import time
import unittest
import _testinternalcapi
import warnings

from test.support import threading_helper


class TestBase(unittest.TestCase):
    pass


def do_race(func1, func2):
    """Run func1() and func2() repeatedly in separate threads."""
    n = 1000

    barrier = threading.Barrier(2)

    def repeat(func):
        barrier.wait()
        for _i in range(n):
            func()

    threads = [
        threading.Thread(target=functools.partial(repeat, func1)),
        threading.Thread(target=functools.partial(repeat, func2)),
    ]
    for thread in threads:
        thread.start()
    for thread in threads:
        thread.join()


@threading_helper.requires_working_threading()
class TestRaces(TestBase):
    def test_racing_cell_set(self):
        """Test cell object gettr/settr properties."""

        def nested_func():
            x = 0

            def inner():
                nonlocal x
                x += 1

        # This doesn't race because LOAD_DEREF and STORE_DEREF on the
        # cell object use critical sections.
        do_race(nested_func, nested_func)

        def nested_func2():
            x = 0

            def inner():
                y = x
                frame = sys._getframe(1)
                frame.f_locals["x"] = 2

            return inner

        def mutate_func2():
            inner = nested_func2()
            cell = inner.__closure__[0]
            old_value = cell.cell_contents
            cell.cell_contents = 1000
            time.sleep(0)
            cell.cell_contents = old_value
            time.sleep(0)

        # This revealed a race with cell_set_contents() since it was missing
        # the critical section.
        do_race(nested_func2, mutate_func2)

    def test_racing_cell_cmp_repr(self):
        """Test cell object compare and repr methods."""

        def nested_func():
            x = 0
            y = 0

            def inner():
                return x + y

            return inner.__closure__

        cell_a, cell_b = nested_func()

        def mutate():
            cell_a.cell_contents += 1

        def access():
            cell_a == cell_b
            s = repr(cell_a)

        # cell_richcompare() and cell_repr used to have data races
        do_race(mutate, access)

    def test_racing_load_super_attr(self):
        """Test (un)specialization of LOAD_SUPER_ATTR opcode."""

        class C:
            def __init__(self):
                try:
                    super().__init__
                    super().__init__()
                except RuntimeError:
                    pass  #  happens if __class__ is replaced with non-type

        def access():
            C()

        def mutate():
            # Swap out the super() global with a different one
            real_super = super
            globals()["super"] = lambda s=1: s
            time.sleep(0)
            globals()["super"] = real_super
            time.sleep(0)
            # Swap out the __class__ closure value with a non-type
            cell = C.__init__.__closure__[0]
            real_class = cell.cell_contents
            cell.cell_contents = 99
            time.sleep(0)
            cell.cell_contents = real_class

        # The initial PR adding specialized opcodes for LOAD_SUPER_ATTR
        # had some races (one with the super() global changing and one
        # with the cell binding being changed).
        do_race(access, mutate)

    def test_racing_to_bool(self):

        seq = [1]

        class C:
            def __bool__(self):
                return False

        def access():
            if seq:
                return 1
            else:
                return 2

        def mutate():
            nonlocal seq
            seq = [1]
            time.sleep(0)
            seq = C()
            time.sleep(0)

        do_race(access, mutate)

    def test_racing_store_attr_slot(self):
        class C:
            __slots__ = ['x', '__dict__']

        c = C()

        def set_slot():
            for i in range(10):
                c.x = i
            time.sleep(0)

        def change_type():
            def set_x(self, x):
                pass

            def get_x(self):
                pass

            C.x = property(get_x, set_x)
            time.sleep(0)
            del C.x
            time.sleep(0)

        do_race(set_slot, change_type)

        def set_getattribute():
            C.__getattribute__ = lambda self, x: x
            time.sleep(0)
            del C.__getattribute__
            time.sleep(0)

        do_race(set_slot, set_getattribute)

    def test_racing_store_attr_instance_value(self):
        class C:
            pass

        c = C()

        def set_value():
            for i in range(100):
                c.x = i

        set_value()

        def read():
            x = c.x

        def mutate():
            # Adding a property for 'x' should unspecialize it.
            C.x = property(lambda self: None, lambda self, x: None)
            time.sleep(0)
            del C.x
            time.sleep(0)

        do_race(read, mutate)

    def test_racing_store_attr_with_hint(self):
        class C:
            pass

        c = C()
        for i in range(29):
            setattr(c, f"_{i}", None)

        def set_value():
            for i in range(100):
                c.x = i

        set_value()

        def read():
            x = c.x

        def mutate():
            # Adding a property for 'x' should unspecialize it.
            C.x = property(lambda self: None, lambda self, x: None)
            time.sleep(0)
            del C.x
            time.sleep(0)

        do_race(read, mutate)

    def make_shared_key_dict(self):
        class C:
            pass

        a = C()
        a.x = 1
        return a.__dict__

    def test_racing_store_attr_dict(self):
        """Test STORE_ATTR with various dictionary types."""
        class C:
            pass

        c = C()

        def set_value():
            for i in range(20):
                c.x = i

        def mutate():
            nonlocal c
            c.x = 1
            self.assertTrue(_testinternalcapi.has_inline_values(c))
            for i in range(30):
                setattr(c, f"_{i}", None)
            self.assertFalse(_testinternalcapi.has_inline_values(c.__dict__))
            c.__dict__ = self.make_shared_key_dict()
            self.assertTrue(_testinternalcapi.has_split_table(c.__dict__))
            c.__dict__[1] = None
            self.assertFalse(_testinternalcapi.has_split_table(c.__dict__))
            c = C()

        do_race(set_value, mutate)

    def test_racing_recursion_limit(self):
        def something_recursive():
            def count(n):
                if n > 0:
                    return count(n - 1) + 1
                return 0

            count(50)

        def set_recursion_limit():
            for limit in range(100, 200):
                sys.setrecursionlimit(limit)

        do_race(something_recursive, set_recursion_limit)


@threading_helper.requires_working_threading()
class TestWarningsRaces(TestBase):
    def setUp(self):
        self.saved_filters = warnings.filters[:]
        warnings.resetwarnings()
        # Add multiple filters to the list to increase odds of race.
        for lineno in range(20):
            warnings.filterwarnings('ignore', message='not matched', category=Warning, lineno=lineno)
        # Override showwarning() so that we don't actually show warnings.
        def showwarning(*args):
            pass
        warnings.showwarning = showwarning

    def tearDown(self):
        warnings.filters[:] = self.saved_filters
        warnings.showwarning = warnings._showwarning_orig

    def test_racing_warnings_filter(self):
        # Modifying the warnings.filters list while another thread is using
        # warn() should not crash or race.
        def modify_filters():
            time.sleep(0)
            warnings.filters[:] = [('ignore', None, UserWarning, None, 0)]
            time.sleep(0)
            warnings.filters[:] = self.saved_filters

        def emit_warning():
            warnings.warn('dummy message', category=UserWarning)

        do_race(modify_filters, emit_warning)


if __name__ == "__main__":
    unittest.main()

### File: test_resource.py

In [ ]:
import unittest
from test.support import import_helper, threading_helper

resource = import_helper.import_module("resource")


NTHREADS = 10
LOOP_PER_THREAD = 1000


@threading_helper.requires_working_threading()
class ResourceTest(unittest.TestCase):
    @unittest.skipUnless(hasattr(resource, "getrusage"), "needs getrusage")
    @unittest.skipUnless(
        hasattr(resource, "RUSAGE_THREAD"), "needs RUSAGE_THREAD"
    )
    def test_getrusage(self):
        ru_utime_lst = []

        def dummy_work(ru_utime_lst):
            for _ in range(LOOP_PER_THREAD):
                pass

            usage_process = resource.getrusage(resource.RUSAGE_SELF)
            usage_thread = resource.getrusage(resource.RUSAGE_THREAD)
            # Process user time should be greater than thread user time
            self.assertGreater(usage_process.ru_utime, usage_thread.ru_utime)
            ru_utime_lst.append(usage_thread.ru_utime)

        threading_helper.run_concurrently(
            worker_func=dummy_work, args=(ru_utime_lst,), nthreads=NTHREADS
        )

        usage_process = resource.getrusage(resource.RUSAGE_SELF)
        self.assertEqual(len(ru_utime_lst), NTHREADS)
        # Process user time should be greater than sum of all thread user times
        self.assertGreater(usage_process.ru_utime, sum(ru_utime_lst))


if __name__ == "__main__":
    unittest.main()

### File: test_reversed.py

In [ ]:
import unittest
from threading import Barrier, Thread
from test.support import threading_helper

threading_helper.requires_working_threading(module=True)

class TestReversed(unittest.TestCase):

    @threading_helper.reap_threads
    def test_reversed(self):
        # Iterating over the iterator with multiple threads should not
        # emit TSAN warnings
        number_of_iterations = 10
        number_of_threads = 10
        size = 1_000

        barrier = Barrier(number_of_threads)
        def work(r):
            barrier.wait()
            while True:
                try:
                     l = r.__length_hint__()
                     next(r)
                except StopIteration:
                    break
                assert 0 <= l <= size
        x = tuple(range(size))

        for _ in range(number_of_iterations):
            r = reversed(x)
            worker_threads = []
            for _ in range(number_of_threads):
                worker_threads.append(Thread(target=work, args=[r]))
            with threading_helper.start_threads(worker_threads):
                pass
            barrier.reset()

if __name__ == "__main__":
    unittest.main()

### File: test_set.py

In [ ]:
import unittest

from threading import Thread, Barrier
from unittest import TestCase

from test.support import threading_helper


@threading_helper.requires_working_threading()
class TestSet(TestCase):
    def test_repr_clear(self):
        """Test repr() of a set while another thread is calling clear()"""
        NUM_ITERS = 10
        NUM_REPR_THREADS = 10
        barrier = Barrier(NUM_REPR_THREADS + 1)
        s = {1, 2, 3, 4, 5, 6, 7, 8}

        def clear_set():
            barrier.wait()
            s.clear()

        def repr_set():
            barrier.wait()
            set_reprs.append(repr(s))

        for _ in range(NUM_ITERS):
            set_reprs = []
            threads = [Thread(target=clear_set)]
            for _ in range(NUM_REPR_THREADS):
                threads.append(Thread(target=repr_set))
            for t in threads:
                t.start()
            for t in threads:
                t.join()

            for set_repr in set_reprs:
                self.assertIn(set_repr, ("set()", "{1, 2, 3, 4, 5, 6, 7, 8}"))


if __name__ == "__main__":
    unittest.main()

### File: test_slots.py

In [ ]:
import _testcapi
import threading
from test.support import threading_helper
from unittest import TestCase


def run_in_threads(targets):
    """Run `targets` in separate threads"""
    threads = [
        threading.Thread(target=target)
        for target in targets
    ]
    for thread in threads:
        thread.start()
    for thread in threads:
        thread.join()


@threading_helper.requires_working_threading()
class TestSlots(TestCase):

    def test_object(self):
        class Spam:
            __slots__ = [
                "eggs",
            ]

            def __init__(self, initial_value):
                self.eggs = initial_value

        spam = Spam(0)
        iters = 20_000

        def writer():
            for _ in range(iters):
                spam.eggs += 1

        def reader():
            for _ in range(iters):
                eggs = spam.eggs
                assert type(eggs) is int
                assert 0 <= eggs <= iters

        run_in_threads([writer, reader, reader, reader])

    def test_T_BOOL(self):
        spam_old = _testcapi._test_structmembersType_OldAPI()
        spam_new = _testcapi._test_structmembersType_NewAPI()

        def writer():
            for _ in range(1_000):
                # different code paths for True and False
                spam_old.T_BOOL = True
                spam_new.T_BOOL = True
                spam_old.T_BOOL = False
                spam_new.T_BOOL = False

        def reader():
            for _ in range(1_000):
                spam_old.T_BOOL
                spam_new.T_BOOL

        run_in_threads([writer, reader])

    def test_T_BYTE(self):
        spam_old = _testcapi._test_structmembersType_OldAPI()
        spam_new = _testcapi._test_structmembersType_NewAPI()

        def writer():
            for _ in range(1_000):
                spam_old.T_BYTE = 0
                spam_new.T_BYTE = 0

        def reader():
            for _ in range(1_000):
                spam_old.T_BYTE
                spam_new.T_BYTE

        run_in_threads([writer, reader])

    def test_T_UBYTE(self):
        spam_old = _testcapi._test_structmembersType_OldAPI()
        spam_new = _testcapi._test_structmembersType_NewAPI()

        def writer():
            for _ in range(1_000):
                spam_old.T_UBYTE = 0
                spam_new.T_UBYTE = 0

        def reader():
            for _ in range(1_000):
                spam_old.T_UBYTE
                spam_new.T_UBYTE

        run_in_threads([writer, reader])

    def test_T_SHORT(self):
        spam_old = _testcapi._test_structmembersType_OldAPI()
        spam_new = _testcapi._test_structmembersType_NewAPI()

        def writer():
            for _ in range(1_000):
                spam_old.T_SHORT = 0
                spam_new.T_SHORT = 0

        def reader():
            for _ in range(1_000):
                spam_old.T_SHORT
                spam_new.T_SHORT

        run_in_threads([writer, reader])

    def test_T_USHORT(self):
        spam_old = _testcapi._test_structmembersType_OldAPI()
        spam_new = _testcapi._test_structmembersType_NewAPI()

        def writer():
            for _ in range(1_000):
                spam_old.T_USHORT = 0
                spam_new.T_USHORT = 0

        def reader():
            for _ in range(1_000):
                spam_old.T_USHORT
                spam_new.T_USHORT

        run_in_threads([writer, reader])

    def test_T_INT(self):
        spam_old = _testcapi._test_structmembersType_OldAPI()
        spam_new = _testcapi._test_structmembersType_NewAPI()

        def writer():
            for _ in range(1_000):
                spam_old.T_INT = 0
                spam_new.T_INT = 0

        def reader():
            for _ in range(1_000):
                spam_old.T_INT
                spam_new.T_INT

        run_in_threads([writer, reader])

    def test_T_UINT(self):
        spam_old = _testcapi._test_structmembersType_OldAPI()
        spam_new = _testcapi._test_structmembersType_NewAPI()

        def writer():
            for _ in range(1_000):
                spam_old.T_UINT = 0
                spam_new.T_UINT = 0

        def reader():
            for _ in range(1_000):
                spam_old.T_UINT
                spam_new.T_UINT

        run_in_threads([writer, reader])

    def test_T_LONG(self):
        spam_old = _testcapi._test_structmembersType_OldAPI()
        spam_new = _testcapi._test_structmembersType_NewAPI()

        def writer():
            for _ in range(1_000):
                spam_old.T_LONG = 0
                spam_new.T_LONG = 0

        def reader():
            for _ in range(1_000):
                spam_old.T_LONG
                spam_new.T_LONG

        run_in_threads([writer, reader])

    def test_T_ULONG(self):
        spam_old = _testcapi._test_structmembersType_OldAPI()
        spam_new = _testcapi._test_structmembersType_NewAPI()

        def writer():
            for _ in range(1_000):
                spam_old.T_ULONG = 0
                spam_new.T_ULONG = 0

        def reader():
            for _ in range(1_000):
                spam_old.T_ULONG
                spam_new.T_ULONG

        run_in_threads([writer, reader])

    def test_T_PYSSIZET(self):
        spam_old = _testcapi._test_structmembersType_OldAPI()
        spam_new = _testcapi._test_structmembersType_NewAPI()

        def writer():
            for _ in range(1_000):
                spam_old.T_PYSSIZET = 0
                spam_new.T_PYSSIZET = 0

        def reader():
            for _ in range(1_000):
                spam_old.T_PYSSIZET
                spam_new.T_PYSSIZET

        run_in_threads([writer, reader])

    def test_T_FLOAT(self):
        spam_old = _testcapi._test_structmembersType_OldAPI()
        spam_new = _testcapi._test_structmembersType_NewAPI()

        def writer():
            for _ in range(1_000):
                spam_old.T_FLOAT = 0.0
                spam_new.T_FLOAT = 0.0

        def reader():
            for _ in range(1_000):
                spam_old.T_FLOAT
                spam_new.T_FLOAT

        run_in_threads([writer, reader])

    def test_T_DOUBLE(self):
        spam_old = _testcapi._test_structmembersType_OldAPI()
        spam_new = _testcapi._test_structmembersType_NewAPI()

        def writer():
            for _ in range(1_000):
                spam_old.T_DOUBLE = 0.0
                spam_new.T_DOUBLE = 0.0

        def reader():
            for _ in range(1_000):
                spam_old.T_DOUBLE
                spam_new.T_DOUBLE

        run_in_threads([writer, reader])

    def test_T_LONGLONG(self):
        spam_old = _testcapi._test_structmembersType_OldAPI()
        spam_new = _testcapi._test_structmembersType_NewAPI()

        def writer():
            for _ in range(1_000):
                spam_old.T_LONGLONG = 0
                spam_new.T_LONGLONG = 0

        def reader():
            for _ in range(1_000):
                spam_old.T_LONGLONG
                spam_new.T_LONGLONG

        run_in_threads([writer, reader])

    def test_T_ULONGLONG(self):
        spam_old = _testcapi._test_structmembersType_OldAPI()
        spam_new = _testcapi._test_structmembersType_NewAPI()

        def writer():
            for _ in range(1_000):
                spam_old.T_ULONGLONG = 0
                spam_new.T_ULONGLONG = 0

        def reader():
            for _ in range(1_000):
                spam_old.T_ULONGLONG
                spam_new.T_ULONGLONG

        run_in_threads([writer, reader])

    def test_T_CHAR(self):
        spam_old = _testcapi._test_structmembersType_OldAPI()
        spam_new = _testcapi._test_structmembersType_NewAPI()

        def writer():
            for _ in range(1_000):
                spam_old.T_CHAR = "c"
                spam_new.T_CHAR = "c"

        def reader():
            for _ in range(1_000):
                spam_old.T_CHAR
                spam_new.T_CHAR

        run_in_threads([writer, reader])

### File: test_str.py

In [ ]:
import unittest

from itertools import cycle
from threading import Event, Thread
from unittest import TestCase

from test.support import threading_helper

@threading_helper.requires_working_threading()
class TestStr(TestCase):
    def test_racing_join_extend(self):
        '''Test joining a string being extended by another thread'''
        l = []
        ITERS = 100
        READERS = 10
        done_event = Event()
        def writer_func():
            for i in range(ITERS):
                l.extend(map(str, range(i)))
                l.clear()
            done_event.set()
        def reader_func():
            while not done_event.is_set():
                ''.join(l)
        writer = Thread(target=writer_func)
        readers = []
        for x in range(READERS):
            reader = Thread(target=reader_func)
            readers.append(reader)
            reader.start()

        writer.start()
        writer.join()
        for reader in readers:
            reader.join()

    def test_racing_join_replace(self):
        '''
        Test joining a string of characters being replaced with ephemeral
        strings by another thread.
        '''
        l = [*'abcdefg']
        MAX_ORDINAL = 1_000
        READERS = 10
        done_event = Event()

        def writer_func():
            for i, c in zip(cycle(range(len(l))),
                            map(chr, range(128, MAX_ORDINAL))):
                l[i] = c
            done_event.set()

        def reader_func():
            while not done_event.is_set():
                ''.join(l)
                ''.join(l)
                ''.join(l)
                ''.join(l)

        writer = Thread(target=writer_func)
        readers = []
        for x in range(READERS):
            reader = Thread(target=reader_func)
            readers.append(reader)
            reader.start()

        writer.start()
        writer.join()
        for reader in readers:
            reader.join()


if __name__ == "__main__":
    unittest.main()

### File: test_syslog.py

In [ ]:
import unittest
import threading

from test.support import import_helper, threading_helper
from test.support.threading_helper import run_concurrently

syslog = import_helper.import_module("syslog")

NTHREADS = 32

# Similar to Lib/test/test_syslog.py, this test's purpose is to verify that
# the code neither crashes nor leaks.


@threading_helper.requires_working_threading()
class TestSyslog(unittest.TestCase):
    def test_racing_syslog(self):
        def worker():
            """
            The syslog module provides the following functions:
            openlog(), syslog(), closelog(), and setlogmask().
            """
            thread_id = threading.get_ident()
            syslog.openlog(f"thread-id: {thread_id}")
            try:
                for _ in range(5):
                    syslog.syslog("logline")
                    syslog.setlogmask(syslog.LOG_MASK(syslog.LOG_INFO))
                    syslog.syslog(syslog.LOG_INFO, "logline LOG_INFO")
                    syslog.setlogmask(syslog.LOG_MASK(syslog.LOG_ERR))
                    syslog.syslog(syslog.LOG_ERR, "logline LOG_ERR")
                    syslog.setlogmask(syslog.LOG_UPTO(syslog.LOG_DEBUG))
            finally:
                syslog.closelog()

        # Run the worker concurrently to exercise all these syslog functions
        run_concurrently(
            worker_func=worker,
            nthreads=NTHREADS,
        )


if __name__ == "__main__":
    unittest.main()

### File: test_tokenize.py

In [ ]:
import io
import time
import unittest
import tokenize
from functools import partial
from threading import Thread

from test.support import threading_helper


@threading_helper.requires_working_threading()
class TestTokenize(unittest.TestCase):
    def test_tokenizer_iter(self):
        source = io.StringIO("for _ in a:\n  pass")
        it = tokenize._tokenize.TokenizerIter(source.readline, extra_tokens=False)

        tokens = []
        def next_token(it):
            while True:
                try:
                    r = next(it)
                    tokens.append(tokenize.TokenInfo._make(r))
                    time.sleep(0.03)
                except StopIteration:
                    return

        threads = []
        for _ in range(5):
            threads.append(Thread(target=partial(next_token, it)))

        for thread in threads:
            thread.start()

        for thread in threads:
            thread.join()

        expected_tokens = [
            tokenize.TokenInfo(type=1, string='for', start=(1, 0), end=(1, 3), line='for _ in a:\n'),
            tokenize.TokenInfo(type=1, string='_', start=(1, 4), end=(1, 5), line='for _ in a:\n'),
            tokenize.TokenInfo(type=1, string='in', start=(1, 6), end=(1, 8), line='for _ in a:\n'),
            tokenize.TokenInfo(type=1, string='a', start=(1, 9), end=(1, 10), line='for _ in a:\n'),
            tokenize.TokenInfo(type=11, string=':', start=(1, 10), end=(1, 11), line='for _ in a:\n'),
            tokenize.TokenInfo(type=4, string='', start=(1, 11), end=(1, 11), line='for _ in a:\n'),
            tokenize.TokenInfo(type=5, string='', start=(2, -1), end=(2, -1), line='  pass'),
            tokenize.TokenInfo(type=1, string='pass', start=(2, 2), end=(2, 6), line='  pass'),
            tokenize.TokenInfo(type=4, string='', start=(2, 6), end=(2, 6), line='  pass'),
            tokenize.TokenInfo(type=6, string='', start=(2, -1), end=(2, -1), line='  pass'),
            tokenize.TokenInfo(type=0, string='', start=(2, -1), end=(2, -1), line='  pass'),
        ]

        tokens.sort()
        expected_tokens.sort()
        self.assertListEqual(tokens, expected_tokens)


if __name__ == "__main__":
    unittest.main()

### File: test_type.py

In [ ]:
import threading
import unittest

from concurrent.futures import ThreadPoolExecutor
from threading import Thread
from unittest import TestCase

from test.support import threading_helper



NTHREADS = 6
BOTTOM = 0
TOP = 1000
ITERS = 100

class A:
    attr = 1

@threading_helper.requires_working_threading()
class TestType(TestCase):
    def test_attr_cache(self):
        def read(id0):
            for _ in range(ITERS):
                for _ in range(BOTTOM, TOP):
                    A.attr

        def write(id0):
            for _ in range(ITERS):
                for _ in range(BOTTOM, TOP):
                    # Make _PyType_Lookup cache hot first
                    A.attr
                    A.attr
                    x = A.attr
                    x += 1
                    A.attr = x


        with ThreadPoolExecutor(NTHREADS) as pool:
            pool.submit(read, (1,))
            pool.submit(write, (1,))
            pool.shutdown(wait=True)

    def test_attr_cache_consistency(self):
        class C:
            x = 0

        def writer_func():
            for _ in range(3000):
                C.x
                C.x
                C.x += 1

        def reader_func():
            for _ in range(3000):
                # We should always see a greater value read from the type than the
                # dictionary
                a = C.__dict__['x']
                b = C.x
                self.assertGreaterEqual(b, a)

        self.run_one(writer_func, reader_func)

    def test_attr_cache_consistency_subclass(self):
        class C:
            x = 0

        class D(C):
            pass

        def writer_func():
            for _ in range(3000):
                D.x
                D.x
                C.x += 1

        def reader_func():
            for _ in range(3000):
                # We should always see a greater value read from the type than the
                # dictionary
                a = C.__dict__['x']
                b = D.x
                self.assertGreaterEqual(b, a)

        self.run_one(writer_func, reader_func)

    def test___class___modification(self):
        loops = 200

        class Foo:
            pass

        class Bar:
            pass

        thing = Foo()
        def work():
            foo = thing
            for _ in range(loops):
                foo.__class__ = Bar
                type(foo)
                foo.__class__ = Foo
                type(foo)


        threads = []
        for i in range(NTHREADS):
            thread = threading.Thread(target=work)
            thread.start()
            threads.append(thread)

        for thread in threads:
            thread.join()

    def test_object_class_change(self):
        class Base:
            def __init__(self):
                self.attr = 123
        class ClassA(Base):
            pass
        class ClassB(Base):
            pass

        obj = ClassA()
        # keep reference to __dict__
        d = obj.__dict__
        obj.__class__ = ClassB


    def test_name_change(self):
        class Foo:
            pass

        def writer():
            for _ in range(1000):
                Foo.__name__ = 'Bar'

        def reader():
            for _ in range(1000):
                Foo.__name__

        self.run_one(writer, reader)

    def run_one(self, writer_func, reader_func):
        barrier = threading.Barrier(NTHREADS)

        def wrap_target(target):
            def wrapper():
                barrier.wait()
                target()
            return wrapper

        writer = Thread(target=wrap_target(writer_func))
        readers = []
        for x in range(NTHREADS - 1):
            reader = Thread(target=wrap_target(reader_func))
            readers.append(reader)
            reader.start()

        writer.start()
        writer.join()
        for reader in readers:
            reader.join()

if __name__ == "__main__":
    unittest.main()

### File: test_uuid.py

In [ ]:
import os
import unittest

from test.support import import_helper, threading_helper
from test.support.threading_helper import run_concurrently
from uuid import SafeUUID

c_uuid = import_helper.import_module("_uuid")

NTHREADS = 10
UUID_PER_THREAD = 1000


@threading_helper.requires_working_threading()
class UUIDTests(unittest.TestCase):
    @unittest.skipUnless(os.name == "posix", "POSIX only")
    def test_generate_time_safe(self):
        uuids = []

        def worker():
            local_uuids = []
            for _ in range(UUID_PER_THREAD):
                uuid, is_safe = c_uuid.generate_time_safe()
                self.assertIs(type(uuid), bytes)
                self.assertEqual(len(uuid), 16)
                # Collect the UUID only if it is safe. If not, we cannot ensure
                # UUID uniqueness. According to uuid_generate_time_safe() man
                # page, it is theoretically possible for two concurrently
                # running processes to generate the same UUID(s) if the return
                # value is not 0.
                if is_safe == SafeUUID.safe:
                    local_uuids.append(uuid)

            # Merge all safe uuids
            uuids.extend(local_uuids)

        run_concurrently(worker_func=worker, nthreads=NTHREADS)
        self.assertEqual(len(uuids), len(set(uuids)))

    @unittest.skipUnless(os.name == "nt", "Windows only")
    def test_UuidCreate(self):
        uuids = []

        def worker():
            local_uuids = []
            for _ in range(UUID_PER_THREAD):
                uuid = c_uuid.UuidCreate()
                self.assertIs(type(uuid), bytes)
                self.assertEqual(len(uuid), 16)
                local_uuids.append(uuid)

            # Merge all uuids
            uuids.extend(local_uuids)

        run_concurrently(worker_func=worker, nthreads=NTHREADS)
        self.assertEqual(len(uuids), len(set(uuids)))


if __name__ == "__main__":
    unittest.main()

### File: test_zip.py

In [ ]:
import unittest
from threading import Thread

from test.support import threading_helper


class ZipThreading(unittest.TestCase):
    @staticmethod
    def work(enum):
        while True:
            try:
                next(enum)
            except StopIteration:
                break

    @threading_helper.reap_threads
    @threading_helper.requires_working_threading()
    def test_threading(self):
        number_of_threads = 8
        number_of_iterations = 8
        n = 40_000
        enum = zip(range(n), range(n))
        for _ in range(number_of_iterations):
            worker_threads = []
            for ii in range(number_of_threads):
                worker_threads.append(
                    Thread(
                        target=self.work,
                        args=[
                            enum,
                        ],
                    )
                )
            for t in worker_threads:
                t.start()
            for t in worker_threads:
                t.join()


if __name__ == "__main__":
    unittest.main()